# Figure — Intrinsic dimension through mouse reprogramming

**Dataset** [GSE115943](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE115943) — mouse OSKM reprogramming, serum arm.
Schiebinger G, Shu J, Tabaka M, *et al.* Optimal-transport analysis of single-cell gene expression
identifies developmental trajectories in reprogramming. *Cell* 2019;176:928–943.e22. [doi:10.1016/j.cell.2019.01.006](https://doi.org/10.1016/j.cell.2019.01.006)
181,124 cells, 39 time points, day 0 to 18.

Input: `Input_mouse_reprogramming.csv`. Output: `figures/` (600 dpi PNG + vector PDF) and
`source_data/`. Provenance of every score is in `README.md`.



In [ ]:
import os
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize
from scipy.stats import f_oneway, kruskal, pearsonr, spearmanr
from statsmodels.stats.multicomp import pairwise_tukeyhsd

INPUT = "Input_mouse_reprogramming.csv"
FIG   = "figures"
SRC   = "source_data"
os.makedirs(FIG, exist_ok=True)
os.makedirs(SRC, exist_ok=True)

# ----  settings -------------------------------------------------
DPI = 600
mpl.rcParams.update({
    "figure.dpi":        150,      # on-screen
    "savefig.dpi":       DPI,      # on disk
    "savefig.bbox":      "tight",
    "pdf.fonttype":      42,       # embed TrueType, so text stays editable
    "ps.fonttype":       42,
    "font.family":       "sans-serif",
    "font.sans-serif":   ["DejaVu Sans", "Helvetica", "Arial"],
    "axes.linewidth":    0.9,
    "axes.labelsize":    12,
    "axes.titlesize":    13,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
})
sns.set_theme(style="white", context="paper", rc=mpl.rcParams)

# The two names used on every axis of this figure.
# Titles on or off. Journals want bare panels with the caption carrying the text.
SHOW_TITLES = True


def put_title(ax, text, fontsize=13):
    if SHOW_TITLES:
        ax.set_title(text, fontsize=fontsize)


ID_LABEL      = "ID score"
ENTROPY_LABEL = "Signalling entropy"


def minmax(x):
    """Min-max scale to 0-1 over the values given. Linear, so every statistic is unchanged."""
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo)


def save(fig, name):
    """Write one panel at 600 dpi and as vector PDF."""
    fig.savefig(f"{FIG}/{name}.png", dpi=DPI)
    fig.savefig(f"{FIG}/{name}.pdf")
    print(f"saved  {FIG}/{name}.png  +  .pdf")


---

## 1 · Input

| Column | |
|---|---|
| `day` | collection day, 0 to 18 |
| `ID_score` | intrinsic dimension, IDEAS `local_2nn`, k = 600 |
| `signalling_entropy` | SCENT `CompCCAT` |
| `KS_score`, `GS76_score` | the two EMT scoring schemes |
| `KS_epithelial`, `KS_mesenchymal` | the two arms of the KS score |
| `pluripotency` | pluripotency signature score |
| `w_*` | WOT trajectory weights  |
| `P_*` | WOT fate probabilities  |

Each score also has a `_raw` column: the value as the tool returned it. `w` and `P` are
different objects; the trajectory panels use `w`.


In [ ]:
cells = pd.read_csv(INPUT, index_col=0)

FATES = ["iPSC", "Stromal", "Trophoblast", "Neural"]
W = [f"w_{f}" for f in FATES]
P = [f"P_{f}" for f in FATES + ["Other"]]

print(f"{len(cells):,} cells x {cells.shape[1]} columns")
print(f"{cells['day'].nunique()} time points, day {cells['day'].min():g} "
      f"to {cells['day'].max():g}")
print(f"\nfate probabilities sum to 1 per cell: "
      f"{cells[P].sum(axis=1).min():.4f} - {cells[P].sum(axis=1).max():.4f}")
print(f"trajectory weights sum to 1 within each day: "
      f"{cells.groupby('day')[W].sum().values.min():.4f} - "
      f"{cells.groupby('day')[W].sum().values.max():.4f}")

display(cells.head(3))
display(cells[["ID_score_raw", "ID_score",
               "signalling_entropy_raw", "signalling_entropy"]].describe().round(4))


---

## 2 · How the ID score was calculated

**IDEAS** — <https://github.com/maddalenastn/IDEAS>, vendored in `../src/computeID/`.
Staiano M, Cirone N, Biondo M, Osella M, Scialdone A. Intrinsic dimensionality of single-cell
transcriptomic data reveals potency landscapes during cell reprogramming. *bioRxiv* 2025.
[doi:10.1101/2025.07.21.665922](https://doi.org/10.1101/2025.07.21.665922)

`local_2nn`, k = 600 — the two-nearest-neighbour estimator applied inside each cell's
own neighbourhood. IDEAS normalises to 10⁶ counts itself, so the input is **raw counts**.
`id_score=False` returns the raw ID; the 0–1 scaling happens here, not there.

Verbatim from `../src/run_scripts/run_mouse_ideas.py` — run separately, in the `ideas`
environment. This notebook does not re-run it.

```python
import sys

sys.path.insert(0, "/home/csb/sampoorn/IDEAS")

import anndata as ad
from computeID.functions_IDEAS import compute_ID

INPUT = "/home/csb/sampoorn/mouse_reprogramming/IDEAS_input_mouse_reprogramming.h5ad"

OUTPUT_CSV = "/home/csb/sampoorn/mouse_reprogramming/IDEAS_local_2NN_mouse_reprogramming.csv"
OUTPUT_PKL = "/home/csb/sampoorn/mouse_reprogramming/IDEAS_local_2NN_mouse_reprogramming.pkl"
OUTPUT_DONE = "/home/csb/sampoorn/mouse_reprogramming/IDEAS_FINISHED.txt"

print("Loading AnnData...", flush=True)

adata = ad.read_h5ad(INPUT)

print(adata, flush=True)
print("Shape:", adata.shape, flush=True)
print("X type:", type(adata.X), flush=True)
print("X dtype:", adata.X.dtype, flush=True)

print("Starting IDEAS local_2nn...", flush=True)

id_result = compute_ID(
    adata,
    method="local_2nn",
    roots=None,
    n_neighbors=600,
    normalization=True,
    norm_sum=1e6,
    full_output=True,
    id_score=False
)

print("IDEAS calculation finished.", flush=True)
print(id_result, flush=True)

id_result.to_csv(OUTPUT_CSV)
id_result.to_pickle(OUTPUT_PKL)

print("CSV saved:", OUTPUT_CSV, flush=True)
print("PKL saved:", OUTPUT_PKL, flush=True)

with open(OUTPUT_DONE, "w") as f:
    f.write("IDEAS local_2nn calculation completed successfully.\n")

print("DONE.", flush=True)
```


---

## 3 · How the signalling entropy was calculated

**CCAT** — the fast approximation to signalling entropy — computed with `SCENT::CompCCAT`.

Method: Teschendorff AE, Enver T. Single-cell entropy for accurate estimation of
differentiation potency from a cell's transcriptome. *Nat Commun* 2017;8:15599.
[doi:10.1038/ncomms15599](https://doi.org/10.1038/ncomms15599)

Implementation used here: <https://github.com/Zaoqu-Liu/SCENT> — an Rcpp-optimised fork of
the authors' original package (<https://github.com/aet21/SCENT>). Vendored in `../src/scent/`.

Provenance as recorded in `adata.uns["SCENT_CCAT"]` of the annotated object:

| | |
|---|---|
| version / commit | 2.0.0 / `8a3a80953befa50b4320249d28e1ce0afb6c5e06` |
| method | `CompCCAT` |
| PPI network | `net13Jun12` |
| genes used | 15,225 |
| orthology | mouse symbol → human Entrez via MyGene/HomoloGene, strict 1:1 |
| input layer | `counts` |

```r
library(SCENT)
data(net13Jun12)

integ <- DoIntegPPI(exp.m = logcounts, ppiA.m = net13Jun12.m)
ccat  <- CompCCAT(exp = integ$expMC, ppiA = integ$adjMC)
```


---

## 4 · Panel 1 — ID score across reprogramming time

Days 0–7.5, adjacent time points pooled into eight bins. Orange dashed line is the bin mean.
One-way ANOVA across the bins.


In [ ]:
group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}
order = ["0–0.5", "1–1.5", "2–2.5", "3–3.5",
         "4–4.5", "5–5.5", "6–6.5", "7–7.5"]


def time_binned(score, lo=0, hi=7.5):
    """Cells in the day window, with the eight time bins and the score scaled 0-1."""
    d = cells[["day", f"{score}_raw"]].copy()
    d = d[d["day"].between(lo, hi)].dropna()
    d["time_group"] = d["day"].map(group_map)
    d = d.dropna(subset=["time_group"])
    d[score] = minmax(d[f"{score}_raw"])
    return d


def violin_over_time(d, score, ylabel, title, name):
    groups = [d.loc[d["time_group"] == g, score] for g in order]
    F, p_anova = f_oneway(*groups)
    counts = d.groupby("time_group", observed=True).size().reindex(order)
    means  = d.groupby("time_group", observed=True)[score].mean().reindex(order)

    print(f"{ylabel} — one-way ANOVA: F = {F:.3f}, p = {p_anova:.3e}   n = {len(d):,}")

    fig, ax = plt.subplots(figsize=(12, 8))

    sns.violinplot(data=d, x="time_group", y=score, order=order,
                   inner=None, cut=0, linewidth=1.2, color="#3274A1", ax=ax)

    sns.boxplot(data=d, x="time_group", y=score, order=order,
                width=0.15, showfliers=False,
                boxprops=dict(facecolor="white", edgecolor="black", linewidth=1.1),
                whiskerprops=dict(color="black", linewidth=1.1),
                capprops=dict(color="black", linewidth=1.1),
                medianprops=dict(color="black", linewidth=1.5), ax=ax)

    ax.plot(np.arange(len(order)), means.values, color="#ff7f0e", linestyle="--",
            linewidth=1.8, marker="o", markersize=5, label="Mean trajectory", zorder=5)

    p_text = "p < 0.001" if p_anova < 0.001 else f"p = {p_anova:.3f}"
    ax.text(0.02, 0.97, f"One-way ANOVA\nF = {F:.1f}, {p_text}", transform=ax.transAxes,
            ha="left", va="top", fontsize=10,
            bbox=dict(facecolor="white", edgecolor="black", linewidth=0.8,
                      boxstyle="round,pad=0.4"))

    for i, g in enumerate(order):
        ax.text(i, -0.075, f"n = {counts.loc[g]:,}", transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9)

    ax.set_xlabel("Reprogramming time (days)", fontsize=13, labelpad=38)
    ax.set_ylabel(ylabel, fontsize=13)
    put_title(ax, title, 14)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(frameon=False, loc="upper right")
    sns.despine(ax=ax)
    fig.subplots_adjust(bottom=0.20, top=0.93, left=0.10, right=0.98)
    save(fig, name)
    plt.show()

    (d.groupby("time_group", observed=True)[score]
       .agg(["count", "mean", "median", "std"]).reindex(order)
       .to_csv(f"{SRC}/{name}.csv"))
    return d, F, p_anova


id_bins, F_id, p_id = violin_over_time(
    time_binned("ID_score"), "ID_score",
    ID_LABEL, f"{ID_LABEL} across reprogramming time", "panel01_id_across_time")


---

## 5 · Panels 2–4 — score correlations

Each point is a cell, coloured by collection day, with the OLS line in black.


In [ ]:
CORR = []


def scatter_by_day(subset, x, y, xlabel, ylabel, title, subset_label, name,
                   vmin=None, vmax=None, cbar_ticks=None, alpha=0.22):
    d = subset[["day", f"{x}_raw", f"{y}_raw"]].dropna().copy()
    d[x] = minmax(d[f"{x}_raw"])
    d[y] = minmax(d[f"{y}_raw"])

    r, pval = pearsonr(d[x], d[y])
    rho, _  = spearmanr(d[x], d[y])
    slope, intercept = np.polyfit(d[x], d[y], 1)
    x_line = np.linspace(0, 1, 200)

    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(d[x], d[y], c=d["day"], cmap="viridis",
                    norm=Normalize(vmin=d["day"].min() if vmin is None else vmin,
                                   vmax=d["day"].max() if vmax is None else vmax),
                    s=4, alpha=alpha, linewidths=0, rasterized=True)
    ax.plot(x_line, slope * x_line + intercept, color="black", linewidth=2)

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Reprogramming time (days)", fontsize=12)
    if cbar_ticks is not None:
        cbar.set_ticks(cbar_ticks)

    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    put_title(ax, title, 14)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

    # p underflows the double-precision floor at these sample sizes
    p_txt = "p < 1e-300" if pval == 0 else f"p = {pval:.2e}"
    ax.text(0.03, 0.97, f"{subset_label}; n = {len(d):,}\nPearson r = {r:.3f}\n{p_txt}",
            transform=ax.transAxes, ha="left", va="top", fontsize=10,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.9))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    save(fig, name)
    plt.show()

    print(f"{name}: n = {len(d):,}  Pearson r = {r:.4f}  Spearman rho = {rho:.4f}  "
          f"p = {pval:.3e}")
    CORR.append({"panel": name, "x": x, "y": y, "subset": subset_label,
                 "n": len(d), "pearson_r": r, "spearman_rho": rho, "p": pval})


early = cells[cells["day"].between(0, 7.5)]
late  = cells[cells["day"] >= 10.5]
TICKS = [0, 1, 2, 3, 4, 5, 6, 7, 7.5]

scatter_by_day(early, "KS_epithelial", "KS_mesenchymal",
               "KS epithelial score", "KS mesenchymal score",
               "KS epithelial vs KS mesenchymal scores",
               "Days 0–7.5", "panel02_ks_epi_vs_mes_early",
               vmin=0, vmax=7.5, cbar_ticks=TICKS)

scatter_by_day(early, "KS_score", "GS76_score",
               "KS score", "76GS score",
               "KS score vs 76GS score",
               "Days 0–7.5", "panel03_ks_vs_gs76_early",
               vmin=0, vmax=7.5, cbar_ticks=TICKS)

scatter_by_day(cells, "ID_score", "signalling_entropy",
               ID_LABEL, ENTROPY_LABEL,
               f"{ENTROPY_LABEL} vs {ID_LABEL}",
               "All cells", "panel04_entropy_vs_id_all")


---

## 6 · Panel 5 — ID score, pairwise Tukey HSD


In [ ]:
def tukey_heatmap(d, score, title, name):
    tk = pairwise_tukeyhsd(endog=d[score], groups=d["time_group"], alpha=0.05)
    tk_df = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
    tk_df["p-adj"] = tk_df["p-adj"].astype(float)
    tk_df.to_csv(f"{SRC}/{name}.csv", index=False)

    m = pd.DataFrame(np.ones((len(order), len(order))), index=order, columns=order)
    for _, row in tk_df.iterrows():
        m.loc[row["group1"], row["group2"]] = row["p-adj"]
        m.loc[row["group2"], row["group1"]] = row["p-adj"]

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(m, mask=np.triu(np.ones_like(m, dtype=bool)), cmap="viridis_r",
                vmin=0, vmax=0.05, annot=True, fmt=".2e", square=True,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "Tukey-adjusted p-value"}, ax=ax)
    put_title(ax, title, 13)
    ax.set_xlabel("Reprogramming time (days)")
    ax.set_ylabel("Reprogramming time (days)")
    fig.tight_layout()
    save(fig, name)
    plt.show()

    print("adjacent time bins:")
    for i in range(len(order) - 1):
        a, b = order[i], order[i + 1]
        row = tk_df[((tk_df.group1 == a) & (tk_df.group2 == b)) |
                    ((tk_df.group1 == b) & (tk_df.group2 == a))]
        print(f"  {a} vs {b}:  p-adj = {float(row.iloc[0]['p-adj']):.3e}")
    return tk_df


tukey_id = tukey_heatmap(id_bins, "ID_score",
                         f"{ID_LABEL}: pairwise Tukey HSD comparisons",
                         "panel05_id_tukey")


---

## 7 · Panels 6–7 — signalling entropy across time, and its Tukey HSD


In [ ]:
ent_bins, F_ent, p_ent = violin_over_time(
    time_binned("signalling_entropy"), "signalling_entropy",
    ENTROPY_LABEL, f"{ENTROPY_LABEL} across reprogramming time",
    "panel06_entropy_across_time")


In [ ]:
tukey_ent = tukey_heatmap(ent_bins, "signalling_entropy",
                          f"{ENTROPY_LABEL}: pairwise Tukey HSD comparisons",
                          "panel07_entropy_tukey")


---

## 8 · Panels 8–9 — along the Waddington-OT trajectories

Day 10.5 onwards. For fate $k$ and day $t$, the mean over cells with non-zero ancestor
weight, weighted by that weight:

$$\mu_{k,t} = \frac{\sum_i w_{ik}\,s_i}{\sum_i w_{ik}}$$

`n_eff = (Σw)² / Σw²` is the effective sample size behind each point.


In [ ]:
wot = cells[cells["day"] >= 10.5].copy()
for s in ["ID_score", "signalling_entropy", "KS_epithelial", "KS_mesenchymal"]:
    wot[s] = minmax(wot[f"{s}_raw"])


def wot_trajectory(data, score):
    """WOT-weighted mean of a score, per fate per day."""
    rows = []
    for day in sorted(data["day"].dropna().unique()):
        day_data = data[data["day"] == day]
        for fate in FATES:
            wcol = f"w_{fate}"
            t = day_data[[score, wcol]].dropna()
            t = t[t[wcol] > 0]                     # cells on this trajectory
            if len(t) == 0:
                continue
            w = t[wcol].to_numpy()
            rows.append({"day": day, "fate": fate,
                         "weighted_mean": np.average(t[score].to_numpy(), weights=w),
                         "n_cells": len(t),
                         "n_eff": w.sum() ** 2 / (w ** 2).sum()})
    return pd.DataFrame(rows)


def plot_trajectories(traj, ylabel, title, name):
    fig, ax = plt.subplots(figsize=(10, 5))
    for fate in FATES:
        t = traj[traj["fate"] == fate].sort_values("day")
        ax.plot(t["day"], t["weighted_mean"],
                marker="o", markersize=4, linewidth=1.8, label=fate)
    put_title(ax, title, 13)
    ax.set_xlabel("Reprogramming time (days)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    # The scores ARE 0-1; the trajectories occupy a narrow band inside that range,
    # so the axis is left to fit the data. Forcing 0-1 here would flatten the panel
    # and hide the separation between fates, which is what it exists to show.
    ax.legend(title="WOT trajectory", frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    fig.tight_layout()
    save(fig, name)
    plt.show()


id_wot = wot_trajectory(wot, "ID_score")
plot_trajectories(id_wot, f"WOT-weighted {ID_LABEL}",
                  f"{ID_LABEL} along Waddington-OT trajectories",
                  "panel08_id_trajectories")

print("day 18 endpoints:")
print(id_wot[id_wot["day"] == id_wot["day"].max()]
      .set_index("fate")[["weighted_mean", "n_cells", "n_eff"]].round(3).to_string())


In [ ]:
ent_wot = wot_trajectory(wot, "signalling_entropy")
plot_trajectories(ent_wot, f"WOT-weighted {ENTROPY_LABEL}",
                  f"{ENTROPY_LABEL} along Waddington-OT trajectories",
                  "panel09_entropy_trajectories")


---

## 9 · Panels 10–12 — late reprogramming, days ≥ 10.5


In [ ]:
scatter_by_day(late, "ID_score", "pluripotency",
               ID_LABEL, "Pluripotency score",
               f"{ID_LABEL} vs pluripotency score: late reprogramming",
               "Days ≥10.5", "panel10_id_vs_pluripotency_late", alpha=0.20)

scatter_by_day(late, "KS_epithelial", "KS_mesenchymal",
               "KS epithelial score", "KS mesenchymal score",
               "KS epithelial vs KS mesenchymal, days ≥10.5",
               "Days ≥10.5", "panel11_ks_epi_vs_mes_late", alpha=0.20)

scatter_by_day(late, "KS_score", "GS76_score",
               "KS score", "76GS score",
               "KS vs 76GS, days ≥10.5",
               "Days ≥10.5", "panel12_ks_vs_gs76_late", alpha=0.20)


---

## 10 · Panel 13 — KS phase plot along the trajectories

Weighted epithelial against weighted mesenchymal score, line in time order, arrow on the
last step.


In [ ]:
ks_epi_wot = wot_trajectory(wot, "KS_epithelial")
ks_mes_wot = wot_trajectory(wot, "KS_mesenchymal")

ks_wot = ks_epi_wot.merge(ks_mes_wot, on=["day", "fate"],
                          suffixes=("_epithelial", "_mesenchymal"))

fig, ax = plt.subplots(figsize=(8, 7))

for fate in FATES:
    t = ks_wot[ks_wot["fate"] == fate].sort_values("day")

    ax.plot(t["weighted_mean_epithelial"], t["weighted_mean_mesenchymal"],
            marker="o", markersize=4, linewidth=2, label=fate)

    # arrow marks the direction of time
    ax.annotate("",
                xy=(t["weighted_mean_epithelial"].iloc[-1],
                    t["weighted_mean_mesenchymal"].iloc[-1]),
                xytext=(t["weighted_mean_epithelial"].iloc[-2],
                        t["weighted_mean_mesenchymal"].iloc[-2]),
                arrowprops=dict(arrowstyle="->", lw=1.5))

ax.set_xlabel("WOT-weighted KS epithelial score", fontsize=12)
ax.set_ylabel("WOT-weighted KS mesenchymal score", fontsize=12)
put_title(ax, "KS phase plot along WOT trajectories")
ax.legend(title="WOT trajectory", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save(fig, "panel13_ks_phase_trajectories")
plt.show()

print("day 18 endpoints:")
print(ks_wot[ks_wot["day"] == ks_wot["day"].max()]
      .set_index("fate")[["weighted_mean_epithelial", "weighted_mean_mesenchymal"]]
      .round(4).to_string())


---

## 11 · Source data


In [ ]:
traj = (id_wot.rename(columns={"weighted_mean": "ID_score"})
        .merge(ent_wot.rename(columns={"weighted_mean": "signalling_entropy"})
               [["day", "fate", "signalling_entropy"]], on=["day", "fate"])
        .merge(ks_epi_wot.rename(columns={"weighted_mean": "KS_epithelial"})
               [["day", "fate", "KS_epithelial"]], on=["day", "fate"])
        .merge(ks_mes_wot.rename(columns={"weighted_mean": "KS_mesenchymal"})
               [["day", "fate", "KS_mesenchymal"]], on=["day", "fate"]))

traj = traj[["day", "fate", "ID_score", "signalling_entropy",
             "KS_epithelial", "KS_mesenchymal", "n_cells", "n_eff"]]
traj.to_csv(f"{SRC}/wot_trajectories.csv", index=False)

pd.DataFrame(CORR).to_csv(f"{SRC}/correlations.csv", index=False)

pd.DataFrame([
    {"score": ID_LABEL,      "test": "one-way ANOVA, 8 time bins",
     "F": F_id,  "p": p_id,  "n": len(id_bins)},
    {"score": ENTROPY_LABEL, "test": "one-way ANOVA, 8 time bins",
     "F": F_ent, "p": p_ent, "n": len(ent_bins)},
]).to_csv(f"{SRC}/anova.csv", index=False)

print("source data written to", SRC)
print("\nfigures written:")
for f in sorted(os.listdir(FIG)):
    if f.endswith(".png"):
        print(" ", f)

display(pd.DataFrame(CORR))
display(traj.head())
